# Fase 3: Integración de Datos Empíricos Urbanos (Aforo y Turismo)

Este notebook tiene como objetivo recoger los nuevos conjuntos de datos (Aforo de Peatones y Viviendas de Uso Turístico), normalizarlos y cruzarlos geográficamente o por distrito con el `scoring` general preexistente de los locales (`negocios_scored.csv`).

De este modo pasamos de depender de variables y *proxies* matemáticos calculados a variables **empíricas y reales** medidas por el Ayuntamiento de Madrid, proporcionando una precisión infinitamente mayor en el riesgo y supervivencia del sector HORECA comercial.

In [ ]:
import pandas as pd
import numpy as np
import os

## Carga de la Base de Datos Histórica

In [ ]:
# Cargar el CSV ya cocinado de las Fases 1 y 2
csv_path = '../data/processed/negocios_scored.csv'

print(f'Cargando base principal analítica: {csv_path}')
df_negocios = pd.read_csv(csv_path)

# Mostrar la muestra previa antes de la adición de columnas
df_negocios.head(3)

## Funcion Normalizadora de Tipografías
Dado que vamos a cruzar DataFrames externos (Aforo y Turismo) contra el nuestro, creamos un convertidor estándar de mayúsculas y acentos.

In [ ]:
def normalize_str(s):
    if pd.isna(s): return ''
    return str(s).upper().replace('Á', 'A').replace('É', 'E').replace('Í', 'I').replace('Ó', 'O').replace('Ú', 'U').strip()

# Agregamos la columna de unión principal en la base oficial
df_negocios['distrito_norm'] = df_negocios['desc_distrito_local'].apply(normalize_str)

# Limpiar ejecuciones previas (idempotencia)
for col in ['UNIDADES_VUT', 'score_turismo', 'peatones', 'score_transito']:
    if col in df_negocios.columns:
        df_negocios.drop(columns=[col], inplace=True)

## Conjunto 1: Viviendas de Uso Turístico (VUT) - Airbnbs

Este campo nos permite conocer qué distritos tienen una presión turística elevada. Los locales comerciales enfocados a un "Cliente de Paso Rápido y Poder Adquisitivo Estándar/Turista" querrán maximizar esto.

In [ ]:
turismo_path = '../data/VIVIENDAS_USO_TURISTICO.xlsx'
print('Procesando Viviendas de Uso Turístico...')

df_turismo = pd.read_excel(turismo_path)

# Enriquecimiento y cruce
if 'DISTRITO' in df_turismo.columns:
    df_turismo['distrito_norm'] = df_turismo['DISTRITO'].apply(normalize_str)
    # Agrupamos por distrito y sumamos
    turismo_distrito = df_turismo.groupby('distrito_norm')['UNIDADES_VUT'].sum().reset_index()
    
    # Merge
    df_negocios = pd.merge(df_negocios, turismo_distrito, on='distrito_norm', how='left')
    df_negocios['UNIDADES_VUT'] = df_negocios['UNIDADES_VUT'].fillna(0)
    
    # Normalización del Scoring de 0 a 100
    max_vut = df_negocios['UNIDADES_VUT'].max()
    df_negocios['score_turismo'] = (df_negocios['UNIDADES_VUT'] / max_vut * 100) if max_vut > 0 else 0
else:
    df_negocios['score_turismo'] = 0

print('¡Éxito! VUT cruzadas.')

## Conjunto 2: Aforo de Peatones (Flujo Real de Calles)

Millones de registros para conocer el volumen físico (el paso) de las calles, la gran sangre que nutre las terrazas y negocios a nivel ras de suelo.

In [ ]:
aforo_path = '../data/aforo-peatones.xlsx'
print('Procesando Aforo de Peatones...')

df_aforo = pd.read_excel(aforo_path)

# Enriquecimiento y cruce
if 'distrito' in df_aforo.columns and 'peatones' in df_aforo.columns:
    df_aforo['distrito_norm'] = df_aforo['distrito'].apply(normalize_str)
    
    # Sumar por todo el distrito
    aforo_distrito = df_aforo.groupby('distrito_norm')['peatones'].sum().reset_index()
    
    # Merge
    df_negocios = pd.merge(df_negocios, aforo_distrito, on='distrito_norm', how='left')
    df_negocios['peatones'] = df_negocios['peatones'].fillna(0)
    
    # Normalización del Scoring de 0 a 100
    max_peatones = df_negocios['peatones'].max()
    df_negocios['score_transito'] = (df_negocios['peatones'] / max_peatones * 100) if max_peatones > 0 else 0
else:
    df_negocios['score_transito'] = 0

print('¡Éxito! Aforos cruzados.')

## Volcado y Cierre
Eliminamos variables basura de merge temporal y sobreescribimos la obra oficial en `processed`.

In [ ]:
# Limpieza de temporales
df_negocios.drop(columns=['distrito_norm'], inplace=True, errors='ignore')

# Mostrar columnas finales
print("Nuevas dimensiones de dataset de locales:", df_negocios.shape)
print("Nuevos perfiles incorporados:", [c for c in df_negocios.columns if 'score_turismo' in c or 'score_transito' in c])

print('Guardando base de datos enriquecida...')
df_negocios.to_csv(csv_path, index=False)
print('¡Hecho! Base de datos definitiva lista para ser consumida e introducida en la app.py y recommender.')